In [35]:
!pip install -q transformers datasets seqeval accelerate

In [36]:
import numpy as np
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, set_seed
from seqeval.metrics import precision_score, recall_score, f1_score

set_seed(42)

raw = load_dataset("tomaarsen/conll2003")
label_list = raw["train"].features["ner_tags"].feature.names
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

model_name = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True,
                          is_split_into_words=True)
    all_labels = []
    for i, tags in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        label_ids = []
        for w in word_ids:
            if w is None:
                label_ids.append(-100)
            elif w != prev:
                label_ids.append(tags[w])
            else:
                label_ids.append(-100)
            prev = w
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_ds = raw.map(tokenize_and_align, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)
    true_preds, true_labels = [], []
    for pred_row, label_row in zip(preds, labels):
        p_tags, l_tags = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                p_tags.append(label_list[p])
                l_tags.append(label_list[l])
        true_preds.append(p_tags)
        true_labels.append(l_tags)
    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

In [37]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/ner-model /content/ner-model
!ls /content/ner-model

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
config.json	   ner-model		  tokenizer.json
model.safetensors  tokenizer_config.json  training_args.bin


In [38]:
train_lines = [
    "[Siti Nurhaliza](PER) performed at [Dewan Filharmonik Petronas](ORG) in [Kuala Lumpur](LOC).",
    "Dr. [Ravi Kumar](PER) works at [Hospital Sultanah Aminah](ORG) in [Johor Bahru](LOC).",
    "[Nur Aisyah binti Ahmad](PER) studies at [UTM](ORG).",
    "[Ahmad bin Ali](PER) bekerja di [Maybank](ORG) di [Kuala Lumpur](LOC).",
    "[Tan Mei Ling](PER) moved from [Penang](LOC) to [Kuching](LOC) last year.",
    "Encik [Razak bin Hassan](PER) is the manager at [Tenaga Nasional](ORG).",
    "[Lee Chong Wei](PER) trained at [Bukit Jalil](LOC) for many years.",
    "[Michelle Yeoh](PER) was born in [Ipoh](LOC).",
    "Puan [Fatimah binti Yusof](PER) teaches at [SMK Tun Fatimah](ORG) in [Johor Bahru](LOC).",
    "[AirAsia](ORG) flights from [Kuching](LOC) to [Kota Kinabalu](LOC) are fully booked.",
    "[Muthu a/l Selvam](PER) joined [Petronas](ORG) in [Miri](LOC).",
    "The conference will be held in [Putrajaya](LOC) next month.",
    "[Wong Kah Hou](PER) studies at [Universiti Malaya](ORG).",
    "Saya tinggal di [Skudai](LOC) berhampiran [Universiti Teknologi Malaysia](ORG).",
    "[Nicol David](PER) won the squash title in [Penang](LOC).",
    "Cik [Aina binti Rosli](PER) works at [Bank Negara Malaysia](ORG).",
    "[Grab](ORG) drivers in [Kuala Lumpur](LOC) asked for higher fares.",
    "[Chong Yi Xuan](PER) visited [Sibu](LOC) with her family.",
    "The patient was sent to [Hospital Umum Sarawak](ORG) in [Kuching](LOC).",
    "[Kementerian Kesihatan Malaysia](ORG) mengumumkan kempen baharu.",
    "[Arun Pillai](PER) flew from [Kuala Lumpur](LOC) to [Melaka](LOC).",
    "[Yuna](PER) will perform in [George Town](LOC) on Saturday.",
    "[Farhan bin Ismail](PER) is an engineer at [Proton](ORG) in [Shah Alam](LOC).",
    "Dr. [Lim Soo Hock](PER) joined [Sunway Medical Centre](ORG).",
    "[Nurul Izzah binti Kamal](PER) lives in [Seremban](LOC).",
    "The students from [Politeknik Kuching](ORG) visited [Bako National Park](LOC).",
    "[Kavitha Raj](PER) started a new job at [Shopee](ORG).",
    "[Ali bin Abu](PER) membeli kereta di [Ipoh](LOC).",
    "[Ahmad Zaki bin Roslan](PER) dari [Alor Setar](LOC) bekerja di [Telekom Malaysia](ORG).",
    "We had lunch together yesterday.",
    "The weather in the afternoon was very hot.",
    "Hari ini hujan lebat.",
    "Please submit your report before Friday.",
    "[Lau Hui Min](PER) is the president of the robotics club at [UTM](ORG).",
    "[Sarawak](LOC) is known for its rainforests and wildlife.",
    "[Ng Wei Hong](PER) opened a cafe in [Miri](LOC).",
    "[Hospital Pulau Pinang](ORG) is located in [George Town](LOC).",
    "[Hafiz bin Mahmud](PER) dan [Syafiqah binti Daud](PER) bertemu di [Kuantan](LOC).",
    "The meeting with [CIMB](ORG) is scheduled for Tuesday.",
    "[Priya Nair](PER) graduated from [Universiti Sains Malaysia](ORG).",
    "Encik [Jamal bin Osman](PER) from [Kota Bharu](LOC) called the helpdesk.",
    "[Rina Marlina](PER) works for [Petronas](ORG) in [Bintulu](LOC).",
    "[Khairul Amri](PER) joined [Sime Darby](ORG) in [Kuala Lumpur](LOC).",
    "[Lim Jia Yi](PER) is from [Sarikei](LOC).",
    "[Maxis](ORG) and [Celcom](ORG) offer new data plans.",
    "[Sofea binti Hamid](PER) works at [Hospital Kuala Lumpur](ORG).",
    "The festival in [Kuching](LOC) attracts many tourists.",
    "[Daniel Wong](PER) is studying in [Melbourne](LOC) now.",
    "[Angie Shanon](PER) is studying at [Melbourne University](ORG) now.",
    "[Zainal bin Ariffin](PER) is the headmaster of [SMK Batu Lintang](ORG).",
    "[Gopal Krishnan](PER) lives near [Taman Universiti](LOC) in [Skudai](LOC).",
    "[Aisyah](PER) dan [Hakim](PER) pergi ke [Langkawi](LOC).",
    "[Tan Chee Keong](PER) represents [Universiti Malaysia Sarawak](ORG) in the debate.",
    "[Intan Syafiqah](PER) works at [Khazanah Nasional](ORG).",
    "Thank you for your message.",
    "[Lucas anak Jugah](PER) comes from [Sri Aman](LOC) in [Sarawak](LOC).",
    "[Jessica Ling](PER) visited [Penang](LOC) with colleagues from [Intel](ORG).",
    "[Mohd Rizal bin Salleh](PER) teaches at [Universiti Teknologi MARA](ORG).",
    "[Siew Ling](PER) bought tickets for [Malaysia Airlines](ORG).",
    "[Pooja Devi](PER) will move to [Cyberjaya](LOC) next year.",
    "Kedai itu dibuka pada pukul sembilan pagi.",
    "[Ceylonese](ORG) is a famous restaurant in [Kuching](LOC).",
    "[Amirul Hakim](PER) works at [Intel](ORG) in [Penang](LOC).",
    "[Chua Wei Jian](PER) graduated from [Universiti Kebangsaan Malaysia](ORG).",
    "Dr. [Nadia Farhana](PER) conducts research at [Universiti Malaya](ORG).",
    "[Kumaravel Rajan](PER) moved to [Kuala Lumpur](LOC) for work.",
    "[Nur Amirah](PER) is an employee at [Maybank](ORG) in [Johor Bahru](LOC).",
    "[Fong Jia Wen](PER) visited [Kota Kinabalu](LOC) last month.",
    "[Muhammad Firdaus](PER) studies engineering at [Universiti Teknologi PETRONAS](ORG).",
    "[Saraswathy Devi](PER) works at [Hospital Queen Elizabeth](ORG) in [Kota Kinabalu](LOC).",
    "[Daniel anak Mawan](PER) comes from [Kapit](LOC) in [Sarawak](LOC).",
    "[Siti Hajar](PER) attended a seminar in [Putrajaya](LOC).",
    "[Jason Lim](PER) joined [Deloitte](ORG) in [Kuala Lumpur](LOC).",
    "[Aqilah binti Zainuddin](PER) teaches at [Universiti Pendidikan Sultan Idris](ORG).",
    "[Ramesh Kumar](PER) bought a house in [Shah Alam](LOC).",
    "[Chong Wen Qi](PER) works for [Samsung](ORG) in [Penang](LOC).",
    "[Nur Syafiqah](PER) travelled from [Miri](LOC) to [Kuching](LOC).",
    "[Haziq bin Azman](PER) is studying at [Universiti Malaysia Pahang Al-Sultan Abdullah](ORG).",
    "[Mei Xin](PER) started an internship at [Petronas](ORG).",
    "[Arvind Singh](PER) lives in [Seremban](LOC).",
    "[Farah Nadia](PER) attended a conference organised by [MDEC](ORG).",
    "[Lim Chee Seng](PER) works at [CIMB](ORG) in [Melaka](LOC).",
    "[Aiman Zulkifli](PER) visited [Gunung Mulu National Park](LOC).",
    "[Priyanka Menon](PER) graduated from [Universiti Sains Malaysia](ORG).",
    "[Syed Danish](PER) joined [AirAsia](ORG) as an engineer.",
    "[Lee Xin Yi](PER) moved from [Ipoh](LOC) to [Cyberjaya](LOC).",
    "[Mohd Azlan bin Ismail](PER) works at [Tenaga Nasional](ORG).",
    "[Tan Siew Mei](PER) attended a workshop in [Kuching](LOC).",
    "[Kavinesh Raj](PER) is a researcher at [Malaysian Palm Oil Board](ORG).",
    "[Nur Iman](PER) studies computer science at [Universiti Malaysia Sarawak](ORG).",
    "[Wong Zi Hao](PER) visited [Batu Caves](LOC) during the holidays.",
    "[Hakimah binti Salleh](PER) works at [Bank Negara Malaysia](ORG).",
    "[Faizal bin Rahman](PER) travelled to [Bintulu](LOC) for a business meeting.",
    "[Rachel Tan](PER) joined [Shopee](ORG) in [Kuala Lumpur](LOC).",
    "[Muhammad Danish](PER) attended a lecture at [Universiti Putra Malaysia](ORG).",
    "[Siti Khadijah](PER) is from [Kota Bharu](LOC).",
    "[Arun Kumar](PER) works at [Proton](ORG) in [Shah Alam](LOC).",
    "[Yap Mei Ling](PER) visited [Langkawi](LOC) with her friends.",
    "[Nurul Ain](PER) joined [Maxis](ORG) as a software developer.",
    "[Goh Jun Kai](PER) studies at [Universiti Tunku Abdul Rahman](ORG).",
    "[Amir Syahmi](PER) travelled from [Kuantan](LOC) to [Kuala Terengganu](LOC).",
    "[Deepa Krishnan](PER) works at [Sunway Medical Centre](ORG).",
    "[Faris bin Abdullah](PER) attended a training programme in [Putrajaya](LOC).",
    "[Ng Jia Hui](PER) works for [Intel](ORG) in [Penang](LOC).",
    "[Mohamad Irfan](PER) enrolled at [Universiti Teknologi MARA](ORG).",
    "[Shalini Devi](PER) lives near [Johor Bahru](LOC).",
    "[Lee Zhi Yang](PER) works at [Grab](ORG) in [Kuala Lumpur](LOC).",
    "[Nur Maisarah](PER) visited [Taman Negara](LOC) last summer.",
    "[Kenny Wong](PER) joined [CelcomDigi](ORG) in [Petaling Jaya](LOC).",
    "[Siti Nurul](PER) attended a meeting at [Kementerian Pendidikan Malaysia](ORG).",
    "[Viknesh Raj](PER) moved to [George Town](LOC) last year.",
    "[Alicia Chong](PER) graduated from [Universiti Malaysia Sabah](ORG).",
    "[Nur Afiqah binti Rahman](PER), who graduated from [Universiti Malaya](ORG), recently joined [Petronas](ORG) as a data analyst in [Kuala Lumpur](LOC).",
    "After spending three years at [Intel](ORG) in [Penang](LOC), [Tan Wei Jian](PER) accepted a senior engineering position at [Samsung](ORG) in [Johor Bahru](LOC).",
    "Dr. [Kavitha Raj](PER), a medical researcher at [Universiti Sains Malaysia](ORG), presented her findings at a conference held in [Kuching](LOC).",
    "Although [Muhammad Danish](PER) was born in [Kota Bharu](LOC), he completed his degree at [Universiti Teknologi Malaysia](ORG) before starting work at [Maybank](ORG).",
    "[Lee Xin Hui](PER) and [Arun Kumar](PER), who both studied at [Universiti Putra Malaysia](ORG), travelled to [Cyberjaya](LOC) to attend a technology exhibition organized by [MDEC](ORG).",
    "The delegation from [Universiti Malaysia Sarawak](ORG), led by Professor [Wong Chee Ming](PER), visited several research laboratories in [Penang](LOC) before returning to [Kuching](LOC).",
    "When [Siti Nur Amirah](PER) relocated from [Miri](LOC) to [Kuala Lumpur](LOC), she transferred from [Curtin University Malaysia](ORG) to [Universiti Malaya](ORG).",
    "[Ahmad Firdaus bin Ismail](PER), who previously worked at [Tenaga Nasional](ORG) in [Ipoh](LOC), was appointed regional manager for the company's operations in [Kuantan](LOC).",
    "Representatives from [AirAsia](ORG), [Malaysia Airlines](ORG), and [Batik Air](ORG) attended an aviation conference in [Kuala Lumpur](LOC), where [Daniel Wong](PER) presented a proposal.",
    "After completing her internship at [Deloitte](ORG) in [Kuala Lumpur](LOC), [Priya Nair](PER) received an offer to join [EY](ORG) as a financial consultant.",
    "The research team, which included Dr. [Lim Soo Hock](PER) from [Universiti Kebangsaan Malaysia](ORG), collected environmental data from several rivers in [Sarawak](LOC).",
    "[Chong Yi Xuan](PER), whose family lives in [Sibu](LOC), travelled to [Shah Alam](LOC) after receiving an admission offer from [Universiti Teknologi MARA](ORG).",
    "During the meeting in [Putrajaya](LOC), [Kementerian Kesihatan Malaysia](ORG) representatives discussed a collaboration with [Hospital Umum Sarawak](ORG) and researchers from [Universiti Malaysia Sabah](ORG).",
    "The software development team at [Grab](ORG), which operates across several Malaysian cities, appointed [Mohd Rizal bin Salleh](PER) to oversee its new office in [Johor Bahru](LOC).",
    "Before joining [Proton](ORG) in [Shah Alam](LOC), [Ng Wei Hong](PER) completed an automotive engineering programme at [Universiti Malaysia Pahang Al-Sultan Abdullah](ORG).",
    "[Nurul Izzah binti Kamal](PER), who currently lives in [Seremban](LOC), was selected to represent [Universiti Kebangsaan Malaysia](ORG) at an international competition in [Singapore](LOC).",
    "While attending a workshop organized by [Bank Negara Malaysia](ORG) in [Kuala Lumpur](LOC), [Jason Lim](PER) met financial experts from several international institutions.",
    "The hospital director, Dr. [Suresh Menon](PER), announced that [Hospital Sultanah Aminah](ORG) in [Johor Bahru](LOC) would introduce a new digital appointment system next year.",
    "Following the announcement by [Telekom Malaysia](ORG), [Aina binti Rosli](PER) travelled from [Kota Kinabalu](LOC) to [Kuching](LOC) to attend a regional telecommunications meeting.",
    "[Gopal Krishnan](PER), who teaches computer science at [Universiti Teknologi Malaysia](ORG), collaborated with researchers from [Universiti Sains Malaysia](ORG) on a project based in [Penang](LOC).",
    "Selepas menamatkan pengajian di [Universiti Putra Malaysia](ORG), [Nur Syafiqah binti Ahmad](PER) menerima tawaran pekerjaan daripada [Petronas](ORG) di [Bintulu](LOC).",
    "[Ahmad Zaki bin Roslan](PER), yang berasal dari [Alor Setar](LOC), berpindah ke [Cyberjaya](LOC) selepas dilantik sebagai jurutera di [Telekom Malaysia](ORG).",
    "Walaupun [Kavitha Raj](PER) dibesarkan di [Ipoh](LOC), beliau memilih untuk melanjutkan pelajaran di [Universiti Malaysia Sabah](ORG) sebelum bekerja di [Kota Kinabalu](LOC).",
    "Dalam persidangan yang berlangsung di [Putrajaya](LOC), wakil daripada [Kementerian Pendidikan Malaysia](ORG) bertemu dengan Dr. [Nadia Farhana](PER) untuk membincangkan kerjasama penyelidikan.",
    "[Lim Jia Yi](PER), yang bekerja di [Maybank](ORG) di [Kuala Lumpur](LOC), telah dipilih untuk menyertai program latihan kepimpinan yang dianjurkan oleh syarikat tersebut.",
    "Selepas berpindah dari [Kuching](LOC) ke [Johor Bahru](LOC), [Farhan bin Ismail](PER) menyambung pengajian dalam bidang kejuruteraan di [Universiti Teknologi Malaysia](ORG).",
    "Pasukan penyelidik daripada [Universiti Malaysia Sarawak](ORG), yang diketuai oleh Profesor [Tan Chee Keong](PER), menjalankan kajian biodiversiti di [Taman Negara Bako](LOC).",
    "Ketika menghadiri seminar di [Melaka](LOC), [Siti Hajar](PER) bertemu dengan pegawai daripada [Bank Negara Malaysia](ORG) yang sedang mencari peserta untuk program baharu mereka.",
    "[Muhammad Afiq](PER) dilantik sebagai pengurus projek di [Sime Darby](ORG) selepas menamatkan latihan profesionalnya di [Kuala Lumpur](LOC).",
    "Pihak [Hospital Pulau Pinang](ORG) bekerjasama dengan [Universiti Sains Malaysia](ORG) untuk menjalankan kajian kesihatan yang melibatkan penduduk di sekitar [George Town](LOC).",
    "[Rachel Tan](PER), [Jason Teo](PER), and [Priya Nair](PER) represented [Intel](ORG) at a semiconductor technology exhibition in [Penang](LOC).",
    "A joint research project involving [Petronas](ORG), [Universiti Teknologi PETRONAS](ORG), and [Universiti Malaya](ORG) was launched in [Perak](LOC) under the supervision of Dr. [Ravi Kumar](PER).",
    "The new manufacturing facility established by [Proton](ORG) in [Tanjung Malim](LOC) will recruit graduates from [Universiti Malaysia Pahang Al-Sultan Abdullah](ORG), according to [Farid Hakim](PER).",
    "After meeting [Michelle Yeoh](PER) at an event in [Kuala Lumpur](LOC), the students from [Universiti Kebangsaan Malaysia](ORG) discussed opportunities for young Malaysian filmmakers.",
    "The annual technology forum in [Kuching](LOC) brought together engineers from [Huawei](ORG), [Samsung](ORG), and [Intel](ORG), including [Wong Zi Hao](PER), who represented his research team.",
    "It was Dr. [Suresh Menon](PER) who informed [Hospital Selayang](ORG) that the research team would arrive in [Selangor](LOC) on Monday.",
    "According to [Nur Amirah](PER), the new branch of [CIMB](ORG) in [Kota Kinabalu](LOC) will provide additional services to customers from surrounding districts.",
    "Although the conference was initially scheduled for [Kuala Lumpur](LOC), [MDEC](ORG) later confirmed that [Daniel Wong](PER) would deliver his presentation in [Cyberjaya](LOC).",
    "Not only did [Arun Pillai](PER) represent [Petronas](ORG) at the conference in [Bintulu](LOC), but he also introduced a new energy monitoring system developed by his team."
]

test_lines = [
    "[Lim Wei Jie](PER) joined [Maybank](ORG) in [Kuching](LOC).",
    "[Nurul Huda binti Ismail](PER) studies at [Universiti Putra Malaysia](ORG).",
    "Dr. [Suresh Menon](PER) works at [Hospital Selayang](ORG).",
    "[Chin Kah Wai](PER) moved to [Bintulu](LOC).",
    "[Ahmad Danial](PER) bekerja di [Petronas](ORG) di [Miri](LOC).",
    "[Jenny Tan](PER) flew from [Penang](LOC) to [Kota Kinabalu](LOC) on [AirAsia](ORG).",
    "Puan [Salmah binti Karim](PER) teaches at [SMK Kota Samarahan](ORG).",
    "[Vijay Kumar](PER) lives in [Ipoh](LOC).",
    "[Tengku Aliya](PER) visited [Langkawi](LOC).",
    "[Tenaga Nasional](ORG) announced new plans in [Putrajaya](LOC).",
    "[Ong Siew Lan](PER) is from [Sibu](LOC).",
    "[Harith Iskandar](PER) performed in [Johor Bahru](LOC).",
    "[Zahid bin Rahman](PER) works at [Telekom Malaysia](ORG).",
    "[Grace Lau](PER) studied at [Universiti Malaysia Sabah](ORG).",
    "The students visited [Kuching](LOC) last weekend.",
    "Hari ini [Hafizah binti Ramli](PER) ke [Melaka](LOC).",
    "[Fazura](PER) is from [Kuala Lumpur](LOC).",
    "Please send the form by Monday.",
    "[Chan Yoke Mei](PER) represents [Hospital Sungai Buloh](ORG).",
    "[Zack Lee](PER) opened an office of [Shopee](ORG) in [Cyberjaya](LOC).",
    "[Muhammad Afiq](PER) works at [Deloitte](ORG) in [Kuala Lumpur](LOC).",
    "[Lim Hui En](PER) graduated from [Universiti Kebangsaan Malaysia](ORG).",
    "[Suresh Raj](PER) visited [Kota Bharu](LOC) last weekend.",
    "[Nur Balqis](PER) joined [Maxis](ORG) as a data analyst.",
    "[Chong Kai Wen](PER) studies at [Universiti Teknologi PETRONAS](ORG).",
    "[Arun Rajan](PER) moved from [Melaka](LOC) to [Johor Bahru](LOC).",
    "[Aisyah Farzana](PER) works at [Bank Negara Malaysia](ORG).",
    "[Tan Jia Hui](PER) visited [Gunung Kinabalu](LOC) during the holidays.",
    "[Mohd Hafiz](PER) joined [Grab](ORG) in [Petaling Jaya](LOC).",
    "[Priya Kumar](PER) graduated from [Universiti Malaysia Pahang Al-Sultan Abdullah](ORG).",
    "[Daniel anak Bujang](PER) lives in [Betong](LOC).",
    "[Siti Mariam](PER) attended a conference in [Cyberjaya](LOC).",
    "[Wong Chee Ming](PER) works for [Samsung](ORG) in [Penang](LOC).",
    "[Nur Fatin](PER) studies medicine at [Universiti Malaya](ORG).",
    "[Kavitha Nair](PER) travelled from [Kuching](LOC) to [Miri](LOC).",
    "[Jason Teo](PER) joined [Proton](ORG) in [Shah Alam](LOC).",
    "[Farhan Hakimi](PER) visited [Taman Negara](LOC) with his family.",
    "[Lee Mei Qi](PER) works at [CelcomDigi](ORG).",
    "[Ravi Shankar](PER) attended a seminar organised by [MDEC](ORG) in [Putrajaya](LOC).",
    "[Nur Izzati](PER) graduated from [Universiti Malaysia Sabah](ORG).",
    "[Tan Hui Wen](PER), who graduated from [Universiti Malaysia Sabah](ORG), joined [Maxis](ORG) as a network engineer in [Kota Kinabalu](LOC).",
    "After relocating from [Miri](LOC) to [Kuala Lumpur](LOC), [Mohd Faizal bin Hassan](PER) accepted a position at [Bank Negara Malaysia](ORG).",
    "Dr. [Priya Menon](PER) collaborated with researchers from [Universiti Sains Malaysia](ORG) to investigate water pollution near [George Town](LOC).",
    "The management team at [AirAsia](ORG) appointed [Chong Kai Wen](PER) to coordinate its new operations centre in [Sepang](LOC).",
    "Walaupun berasal dari [Kuantan](LOC), [Nur Aina binti Rahim](PER) melanjutkan pengajian di [Universiti Malaya](ORG) sebelum menyertai [Deloitte](ORG).",
    "Representatives from [Universiti Teknologi MARA](ORG) and [Universiti Kebangsaan Malaysia](ORG) attended a research symposium in [Shah Alam](LOC).",
    "[Ravi Shankar](PER), who previously worked in [Penang](LOC), was transferred to the regional office of [Samsung](ORG) in [Johor Bahru](LOC).",
    "Ketika menghadiri mesyuarat di [Putrajaya](LOC), [Siti Mariam](PER) berbincang dengan pegawai daripada [Kementerian Kesihatan Malaysia](ORG) tentang program kesihatan komuniti.",
    "The engineering department at [Proton](ORG) invited [Lee Zhi Yang](PER) to present his university research at a technical workshop in [Tanjung Malim](LOC).",
    "[Alicia Chong](PER), who lives in [Seremban](LOC), travelled to [Cyberjaya](LOC) to attend a cybersecurity seminar organized by [MDEC](ORG).",
    "Selepas menyertai [Telekom Malaysia](ORG), [Muhammad Irfan](PER) ditugaskan untuk menyelia projek rangkaian komunikasi di [Kuching](LOC).",
    "The delegation led by Professor [Kavitha Nair](PER) from [Universiti Putra Malaysia](ORG) visited a biotechnology research centre in [Bangi](LOC).",
    "[Goh Jun Kai](PER) and [Nur Balqis](PER) represented [Universiti Tunku Abdul Rahman](ORG) at a national robotics competition in [Kuala Lumpur](LOC).",
    "The new research agreement between [Petronas](ORG) and [Universiti Teknologi PETRONAS](ORG) was announced by Dr. [Faridah Hassan](PER) during a ceremony in [Seri Iskandar](LOC).",
    "Although [Daniel anak Mawan](PER) grew up in [Sri Aman](LOC), he moved to [Kuching](LOC) after receiving an engineering offer from [Sarawak Energy](ORG).",
    "[Nurul Huda binti Ismail](PER), who works at [Hospital Sungai Buloh](ORG), attended a medical technology conference in [Petaling Jaya](LOC).",
    "Ketua penyelidik Dr. [Lim Jia Wen](PER) dari [Universiti Malaysia Sarawak](ORG) membentangkan hasil kajian biodiversiti yang dijalankan di [Mulu](LOC).",
    "The partnership involving [CIMB](ORG), [Maybank](ORG), and [Universiti Malaya](ORG) was discussed during a financial technology forum in [Kuala Lumpur](LOC).",
    "[Harith Iskandar](PER) travelled to [Kuching](LOC) to attend an event organized by [Sarawak Tourism Board](ORG), where local entrepreneurs presented their businesses.",
    "Selepas berpindah ke [Melaka](LOC), [Chan Yoke Mei](PER) menyertai [Infineon Technologies](ORG) sebagai jurutera semikonduktor."
]
print(len(train_lines), len(test_lines))

151 60


In [39]:
import re

PATTERN = re.compile(r"\[([^\]]+)\]\((PER|ORG|LOC)\)")

def tokenize(s):
    return re.findall(r"\w+(?:[-']\w+)*|[^\w\s]", s)

def parse(line):
    tokens, tags, pos = [], [], 0
    for m in PATTERN.finditer(line):
        for t in tokenize(line[pos:m.start()]):
            tokens.append(t); tags.append("O")
        for i, t in enumerate(tokenize(m.group(1))):
            tokens.append(t)
            tags.append(("B-" if i == 0 else "I-") + m.group(2))
        pos = m.end()
    for t in tokenize(line[pos:]):
        tokens.append(t); tags.append("O")
    return tokens, tags

def build(lines):
    data = {"tokens": [], "ner_tags": []}
    for line in lines:
        tokens, tags = parse(line)
        data["tokens"].append(tokens)
        data["ner_tags"].append([label2id[t] for t in tags])
    return data

# 1) check for typos such as (PRE) that the parser would not understand
for line in train_lines + test_lines:
    toks, _ = parse(line)
    if any(t in toks for t in ["[", "]", "(", ")"]):
        print("CHECK THIS LINE:", line)

# 2) look at one example
print(list(zip(*parse(train_lines[0]))))

# 3) build the datasets
mal_train = Dataset.from_dict(build(train_lines)).map(tokenize_and_align, batched=True)
mal_test = Dataset.from_dict(build(test_lines)).map(tokenize_and_align, batched=True)

[('Siti', 'B-PER'), ('Nurhaliza', 'I-PER'), ('performed', 'O'), ('at', 'O'), ('Dewan', 'B-ORG'), ('Filharmonik', 'I-ORG'), ('Petronas', 'I-ORG'), ('in', 'O'), ('Kuala', 'B-LOC'), ('Lumpur', 'I-LOC'), ('.', 'O')]


Map:   0%|          | 0/151 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

In [40]:
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)

collator = DataCollatorForTokenClassification(tokenizer)
eval_args = TrainingArguments(output_dir="tmp-eval", per_device_eval_batch_size=32,
                              report_to="none")

def evaluate_on(model, dataset):
    t = Trainer(model=model, args=eval_args, data_collator=collator,
                compute_metrics=compute_metrics)
    return t.evaluate(dataset)

baseline = AutoModelForTokenClassification.from_pretrained("ner-model")
before = evaluate_on(baseline, mal_test)
print("BEFORE (Malaysian test):", before)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Precision,Recall,F1
No log,0.238546,0,0.800000,0.860759,0.829268


BEFORE (Malaysian test): {'eval_loss': 0.23854568600654602, 'eval_precision': 0.8, 'eval_recall': 0.8607594936708861, 'eval_f1': 0.829268292682927}


In [41]:
from transformers import pipeline

def show_predictions(model, lines):
    nlp = pipeline("token-classification", model=model, tokenizer=tokenizer,
                   aggregation_strategy="first")
    for line in lines:
        text = PATTERN.sub(lambda m: m.group(1), line)
        preds = [(e["word"], e["entity_group"]) for e in nlp(text)
                 if e["entity_group"] in {"PER", "ORG", "LOC"}]
        print("GOLD:", line)
        print("PRED:", preds)
        print()

show_predictions(baseline, test_lines)

GOLD: [Lim Wei Jie](PER) joined [Maybank](ORG) in [Kuching](LOC).
PRED: [('Lim Wei Jie', 'PER'), ('Maybank', 'ORG'), ('Kuching', 'LOC')]

GOLD: [Nurul Huda binti Ismail](PER) studies at [Universiti Putra Malaysia](ORG).
PRED: [('Nurul Huda binti Ismail', 'PER'), ('Universiti Putra Malaysia', 'ORG')]

GOLD: Dr. [Suresh Menon](PER) works at [Hospital Selayang](ORG).
PRED: [('Suresh Menon', 'PER'), ('Hospital', 'LOC'), ('Selayang', 'LOC')]

GOLD: [Chin Kah Wai](PER) moved to [Bintulu](LOC).
PRED: [('Chin Kah Wai', 'PER'), ('Bintulu', 'LOC')]

GOLD: [Ahmad Danial](PER) bekerja di [Petronas](ORG) di [Miri](LOC).
PRED: [('Ahmad Danial', 'PER'), ('Petronas di Miri', 'ORG')]

GOLD: [Jenny Tan](PER) flew from [Penang](LOC) to [Kota Kinabalu](LOC) on [AirAsia](ORG).
PRED: [('Jenny Tan', 'PER'), ('Penang', 'LOC'), ('Kota Kinabalu', 'LOC'), ('AirAsia', 'ORG')]

GOLD: Puan [Salmah binti Karim](PER) teaches at [SMK Kota Samarahan](ORG).
PRED: [('Puan Salmah', 'PER'), ('Karim', 'PER'), ('SMK Kota Sam

In [43]:
trainer2.compute_metrics = compute_metrics

after = trainer2.evaluate(mal_test)
print("AFTER  (Malaysian test):", after)

conll_after = trainer2.evaluate(tokenized_ds["test"])
print("AFTER  (CoNLL test)    :", conll_after)

Training Loss,Validation Loss,Step,Precision,Recall,F1
0.005713,0.009678,1035,1.000000,1.000000,1.000000


AFTER  (Malaysian test): {'eval_loss': 0.0096780676394701, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0}


Training Loss,Validation Loss,Step,Precision,Recall,F1
0.005713,0.147915,1035,0.876166,0.898194,0.887043


AFTER  (CoNLL test)    : {'eval_loss': 0.14791464805603027, 'eval_precision': 0.8761658031088083, 'eval_recall': 0.8981940509915014, 'eval_f1': 0.8870431893687708}


In [44]:
def entities(lines):
    ents = set()
    for line in lines:
        for m in PATTERN.finditer(line):
            ents.add((m.group(1).lower(), m.group(2)))
    return ents

tr, te = entities(train_lines), entities(test_lines)
print("exact duplicate sentences:", set(train_lines) & set(test_lines))
print("test entities also in train:", len(te & tr), "of", len(te))
print("new in test:", sorted(te - tr))

exact duplicate sentences: set()
test entities also in train: 57 of 108
new in test: [('ahmad danial', 'PER'), ('aisyah farzana', 'PER'), ('arun rajan', 'PER'), ('bangi', 'LOC'), ('betong', 'LOC'), ('chan yoke mei', 'PER'), ('chin kah wai', 'PER'), ('chong kai wen', 'PER'), ('daniel anak bujang', 'PER'), ('farhan hakimi', 'PER'), ('faridah hassan', 'PER'), ('fazura', 'PER'), ('grace lau', 'PER'), ('gunung kinabalu', 'LOC'), ('hafizah binti ramli', 'PER'), ('harith iskandar', 'PER'), ('hospital sungai buloh', 'ORG'), ('infineon technologies', 'ORG'), ('jenny tan', 'PER'), ('kavitha nair', 'PER'), ('lee mei qi', 'PER'), ('lim hui en', 'PER'), ('lim jia wen', 'PER'), ('lim wei jie', 'PER'), ('mohd faizal bin hassan', 'PER'), ('mohd hafiz', 'PER'), ('muhammad irfan', 'PER'), ('mulu', 'LOC'), ('nur aina binti rahim', 'PER'), ('nur balqis', 'PER'), ('nur fatin', 'PER'), ('nur izzati', 'PER'), ('nurul huda binti ismail', 'PER'), ('ong siew lan', 'PER'), ('priya kumar', 'PER'), ('priya menon',

In [46]:
import torch

# 1) strict test set: sentences with at least one entity, none of them seen in training
def sent_entities(line):
    return {(m.group(1).lower(), m.group(2)) for m in PATTERN.finditer(line)}

strict_lines = [l for l in test_lines
                if sent_entities(l) and not (sent_entities(l) & tr)]
print("strict test sentences:", len(strict_lines), "of", len(test_lines))

strict_test = Dataset.from_dict(build(strict_lines)).map(tokenize_and_align, batched=True)

# 2) simple evaluation function (avoids the Trainer error from before)
def eval_manual(model, dataset):
    model.eval()
    dev = next(model.parameters()).device
    gold, pred = [], []
    for ex in dataset:
        ids = torch.tensor([ex["input_ids"]], device=dev)
        mask = torch.tensor([ex["attention_mask"]], device=dev)
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=mask).logits[0].argmax(-1).tolist()
        g, p = [], []
        for lab, pr in zip(ex["labels"], out):
            if lab != -100:
                g.append(label_list[lab])
                p.append(label_list[pr])
        gold.append(g)
        pred.append(p)
    return {"precision": precision_score(gold, pred),
            "recall": recall_score(gold, pred),
            "f1": f1_score(gold, pred)}

print("BEFORE (strict):", eval_manual(baseline, strict_test))
print("AFTER  (strict):", eval_manual(model2, strict_test))

strict test sentences: 4 of 60


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

BEFORE (strict): {'precision': np.float64(0.7777777777777778), 'recall': np.float64(0.875), 'f1': np.float64(0.823529411764706)}
AFTER  (strict): {'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0)}


In [47]:
from seqeval.metrics.sequence_labeling import get_entities

def predict_tags(model, dataset):
    model.eval()
    dev = next(model.parameters()).device
    gold, pred = [], []
    for ex in dataset:
        ids = torch.tensor([ex["input_ids"]], device=dev)
        mask = torch.tensor([ex["attention_mask"]], device=dev)
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=mask).logits[0].argmax(-1).tolist()
        g = [label_list[l] for l in ex["labels"] if l != -100]
        p = [label_list[o] for l, o in zip(ex["labels"], out) if l != -100]
        gold.append(g)
        pred.append(p)
    return gold, pred

def ent_set(tokens, tags):
    return {(" ".join(tokens[s:e + 1]).lower(), t) for t, s, e in get_entities(tags)}

# entities seen in training (same tokenization as the test side)
seen = set()
for line in train_lines:
    toks, tags = parse(line)
    seen |= ent_set(toks, tags)

def unseen_recall(model, name):
    gold, pred = predict_tags(model, mal_test)
    total, found, missed = 0, 0, []
    for toks, g, p in zip(mal_test["tokens"], gold, pred):
        gold_ents, pred_ents = ent_set(toks, g), ent_set(toks, p)
        for e in gold_ents - seen:
            total += 1
            if e in pred_ents:
                found += 1
            else:
                missed.append(e)
    print(f"{name}: found {found} of {total} unseen entities ({found/total:.3f})")
    return missed

unseen_recall(baseline, "BEFORE")
missed_after = unseen_recall(model2, "AFTER ")
print("still missed after training:", missed_after)

BEFORE: found 50 of 60 unseen entities (0.833)
AFTER : found 60 of 60 unseen entities (1.000)
still missed after training: []


In [48]:
trainer2.save_model("ner-model-malaysian")
tokenizer.save_pretrained("ner-model-malaysian")
!cp -r ner-model-malaysian /content/drive/MyDrive/

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

small self-written test set, same author, about 150 training sentences